# ColabNAS


# Các bước chuẩn bị

### Import thư viện

In [1]:
!pip install --quiet wget
import os
from pathlib import Path

import tensorflow as tf
from tensorflow import keras
import numpy as np

import subprocess
import re
import datetime
import shutil
import glob
import wget
import ssl
import gdown

  Preparing metadata (setup.py) ... done


2026-08-10 14:43:35.921704: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1786373016.125350      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1786373016.194330      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1786373016.705997      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1786373016.706036      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1786373016.706039      57 computation_placer.cc:177] computation placer alr

In [2]:
# disable warnings for cleaner console output
import absl.logging
import warnings
absl.logging.set_verbosity(absl.logging.ERROR)
warnings.filterwarnings("ignore")

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'   

tf.get_logger().setLevel('ERROR')          
tf.autograph.set_verbosity(0)

### Cấp quyền thực thi cho `stm32tflm`

In [3]:
!cp /kaggle/input/datasets/karosvn/stm32tflm-linux/stm32tflm /kaggle/working
!chmod +x /kaggle/working/stm32tflm
!test -x /kaggle/working/stm32tflm && echo "Executable" || echo "Not executable"

Executable


# Cài đặt ColabNAS

In [4]:
FIXED_SEED = 11

In [5]:
# Improve vers
class ColabNAS :
    def __init__(self, max_RAM, max_Flash, max_MACC, path_to_training_set, epochs,
                 val_split, cache=False, input_shape=(50,50,3), save_path='.',
                 path_to_stm32tflm='/kaggle/working/stm32tflm', two_stage=False):        
        self.learning_rate = 1e-3
        self.batch_size = 128
        self.epochs = epochs
        self.model = None

        self.max_MACC = max_MACC
        self.max_Flash = max_Flash
        self.max_RAM = max_RAM
        self.path_to_training_set = path_to_training_set
        self.num_classes = len(next(os.walk(path_to_training_set))[1])
        self.val_split = val_split
        self.cache = cache
        self.input_shape = input_shape
        self.save_path = Path(save_path)
        self.two_stage = two_stage

        self.path_to_trained_models = self.save_path / "trained_models"
        self.path_to_trained_models.mkdir(parents=True, exist_ok=True)
        self.path_to_stm32tflm = Path(path_to_stm32tflm)

        self.augmentation = tf.keras.Sequential([
            tf.keras.layers.RandomFlip("horizontal"),
            tf.keras.layers.RandomRotation(
                0.2,
                fill_mode="constant",
                interpolation="bilinear"
            )
        ])        

        self.load_training_set()


    # k: number of kernels of the first convolutional layer
    # c: number of cells added upon the first convolutional layer
    # pre-processing pipeline not included in MACC computation
    def Model(self, k, c):
        # [Fix Randomness] Fix random seed for weight initialization to guarantee consistent model convergence
        #init = tf.keras.initializers.GlorotUniform(seed=FIXED_SEED)
        kernel_size = (3, 3)
        pool_size = (2, 2)
        pool_strides = (2, 2)

        number_of_cells_limited = False
        number_of_mac = 0

        # inputs = (50, 50, 3) by default
        inputs = keras.Input(shape=self.input_shape)

        # preprocessing pipeline
        x = tf.keras.layers.Rescaling(1./255)(inputs)
        x = tf.keras.layers.BatchNormalization()(x)

        # convolutional base
        n = k
        multiplier = 2

        # first convolutional layer
        # c_in = 3 now
        c_in = self.input_shape[2]
        # x.shape() = (batch_size, 50, 50, n)
        x = keras.layers.Conv2D(n, kernel_size, activation='relu', padding='same')(x)
        # MAC = 3 * (3x3) * (50x50) * n
        number_of_mac += (c_in * kernel_size[0] * kernel_size[1] * x.shape[1] * x.shape[2] * x.shape[3])

        # adding cells
        for i in range(1, c + 1):
            if x.shape[1] <= 1 or x.shape[2] <= 1:
                number_of_cells_limited = True
                break
            n = int(np.ceil(n * multiplier))
            multiplier = multiplier - 2**-i
            # x.shape() = (batch_size, h_old /2, w_old /2, n_old)
            x = keras.layers.MaxPooling2D(pool_size=pool_size, strides=pool_strides, padding='valid')(x)
            # c_in = n_old
            c_in = x.shape[3]
            # x.shape() = (batch_size, h, w, n_new)
            x = keras.layers.Conv2D(n, kernel_size, activation='relu', padding='same')(x)
            # MAC = c_in * (3x3) * (hxw) * n_new
            number_of_mac += (c_in * kernel_size[0] * kernel_size[1] * x.shape[1] * x.shape[2] * x.shape[3])

        # classifier
        # x.shape() = (batch_size, n_last)
        x = keras.layers.GlobalAveragePooling2D()(x)
        # input_shape = n_last
        input_shape = x.shape[1]
        # x.shape() = (batch_size, n_last)
        x = keras.layers.Dense(n, activation='relu')(x)
        number_of_mac += (input_shape * x.shape[1])
        # outputs.shape() = (batch_size, num_classes)
        outputs = keras.layers.Dense(self.num_classes, activation='softmax')(x)
        number_of_mac += (x.shape[1] * outputs.shape[1])

        model = keras.Model(inputs=inputs, outputs=outputs)

        optimizer = tf.keras.optimizers.Adam(learning_rate=self.learning_rate)
        model.compile(optimizer=optimizer,
                loss='categorical_crossentropy',
                metrics=['accuracy'])
        
        #model.summary()
        self.model = model

        return number_of_mac, number_of_cells_limited

    def train(self, epochs, train_mode, finetune_ratio=0.2, lr_decay=100, callbacks=None):
        # if train_mode == "standard":
        history = self.model.fit(
            self.train_ds,
            epochs=epochs,
            validation_data=self.validation_ds,
            validation_freq=1,
            verbose=0,
            callbacks=callbacks
        )
        return history
    
        # elif train_mode == "two-stage":
        #     finetune_epochs = int(finetune_ratio * epochs)
        #     standard_epochs = epochs - finetune_epochs
    
        #     history1 = self.model.fit(
        #         self.train_ds,
        #         epochs=standard_epochs,
        #         validation_data=self.validation_ds,
        #         validation_freq=1,
        #         verbose=0,
        #         callbacks=callbacks
        #     )
    
        #     # ========== Fine-tune with Augmentation ============
        #     finetune_lr = self.learning_rate / lr_decay
        #     optimizer = tf.keras.optimizers.Adam(learning_rate=finetune_lr)
        #     self.model.compile(optimizer=optimizer,
        #             loss='categorical_crossentropy',
        #             metrics=['accuracy'])
    
        #     history2 = self.model.fit(
        #         self.train_ds_aug,
        #         initial_epoch=standard_epochs,
        #         epochs=finetune_epochs,
        #         validation_data=self.validation_ds,
        #         validation_freq=1,
        #         verbose=0,
        #         callbacks=callbacks
        #     )
    
        #     combined_history = {}
        #     for key in history1.history:
        #         combined_history[key] = history1.history[key] + history2.history.get(key, [])
    
        #     merged = tf.keras.callbacks.History()
        #     merged.history = combined_history
        #     return merged
    
        # else:
        #     raise ValueError(f"Invalid train_mode. Got {train_mode}")


    def load_training_set(self):
        if 3 == self.input_shape[2]:
            color_mode = 'rgb'
        elif 1 == self.input_shape[2]:
            color_mode = 'grayscale'

        train_ds = tf.keras.utils.image_dataset_from_directory(
            directory= self.path_to_training_set,
            labels='inferred',
            label_mode='categorical',
            color_mode=color_mode,
            batch_size=self.batch_size,
            image_size=self.input_shape[0:2],
            shuffle=True,
            seed=FIXED_SEED,
            validation_split=self.val_split,
            subset='training'
        )

        validation_ds = tf.keras.utils.image_dataset_from_directory(
            directory= self.path_to_training_set,
            labels='inferred',
            label_mode='categorical',
            color_mode=color_mode,
            batch_size=self.batch_size,
            image_size=self.input_shape[0:2],
            shuffle=True,
            seed=FIXED_SEED,
            validation_split=self.val_split,
            subset='validation'
        )

        AUTOTUNE = tf.data.AUTOTUNE

        if self.cache:
            train_ds = train_ds.cache()
            validation_ds = validation_ds.cache()

        self.train_ds = train_ds.prefetch(AUTOTUNE)

        self.train_ds_aug = (
            train_ds
            .map(
                lambda x, y: (self.augmentation(x, training=True), y),
                num_parallel_calls=AUTOTUNE,
            )
            .prefetch(AUTOTUNE)
        )

        self.validation_ds = validation_ds.prefetch(AUTOTUNE)

    def quantize_model_uint8(self, src_path, des_path):
        def representative_dataset():
            for data in self.train_ds.rebatch(1).take(150):
                yield [tf.dtypes.cast(data[0], tf.float32)]

        model = tf.keras.models.load_model(src_path)
        converter = tf.lite.TFLiteConverter.from_keras_model(model)
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.representative_dataset = representative_dataset
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        converter.inference_input_type = tf.uint8
        converter.inference_output_type = tf.uint8
        tflite_quant_model = converter.convert()

        with open(des_path, "wb") as f:
            f.write(tflite_quant_model)

        # (self.path_to_trained_models / f"{self.model_name}.h5").unlink()

    def evaluate_flash_and_peak_RAM_occupancy(self, src_path, des_path):
        # quantize model to evaluate peak RAM and Flash occupancy
        self.quantize_model_uint8(
            src_path,
            des_path
        )

        # evaluate peak RAM and Flash occupancy using STMicroelectronics' X-CUBE-AI
        proc = subprocess.Popen(
            [self.path_to_stm32tflm, des_path], 
            stdout=subprocess.PIPE
        )
        try:
            outs, errs = proc.communicate(timeout=15)
            Flash, RAM = re.findall(r'\d+', str(outs))
        except subprocess.TimeoutExpired:
            proc.kill()
            outs, errs = proc.communicate()
            print("stm32tflm error")
            exit()

        return int(Flash), int(RAM)

    def evaluate_model_process(self, k, c):
        if k > 0:
            self.model_name = f"k_{k}_c_{c}"
            print(f"\n{self.model_name}\n")
            h5_path = self.path_to_trained_models / f"{self.model_name}.h5"
            tflite_path = self.path_to_trained_models / f"{self.model_name}.tflite"
            checkpoint = tf.keras.callbacks.ModelCheckpoint(
                str(h5_path), monitor='val_accuracy',
                verbose=0, save_best_only=True, save_weights_only=False, mode='auto'
            )
            
            # [Fix Randomness] Reset the global seed for each NAS iteration to ensure every new model architecture starts with the exact same initial weight sequence
            #tf.keras.backend.clear_session()
            #tf.keras.utils.set_random_seed(FIXED_SEED)
            MACC, number_of_cells_limited = self.Model(k, c)
            # One epoch of training must be done before quantization 
            # which is needed to evaluate RAM and Flash occupancy
            self.train(epochs=1, train_mode="standard")
            self.model.save(h5_path)
            Flash, RAM = self.evaluate_flash_and_peak_RAM_occupancy(
                src_path=h5_path,
                des_path=tflite_path
            )
            print(f"\nRAM: {RAM},\tFlash: {Flash},\tMACC: {MACC}\n")
            
            # If sastisfy hardware-constraints
            # RAM, Flash, MACC, maximum cells
            if MACC <= self.max_MACC and RAM <= self.max_RAM and Flash <= self.max_Flash and not number_of_cells_limited:

                if self.two_stage:
                    train_mode = "two-stage"
                else:
                    train_mode = "standard"

                hist = self.train(
                    epochs=self.epochs - 1,
                    train_mode=train_mode,
                    callbacks=[checkpoint]
                )
                
                self.quantize_model_uint8(
                    src_path=h5_path,
                    des_path=tflite_path
                )
                best_acc = max(hist.history['val_accuracy'])
                best_epoch = hist.history['val_accuracy'].index(best_acc) + 1
                print(f"Best val_accuracy: {best_acc:.3f} at epoch {best_epoch}/{len(hist.history['val_accuracy'])}")
    
            return {
                'k': k,
                'c': c if not number_of_cells_limited else "Not feasible",
                'RAM': RAM if RAM <= self.max_RAM else "Outside the upper bound",
                'Flash': Flash if Flash <= self.max_Flash else "Outside the upper bound",
                'MACC': MACC if MACC <= self.max_MACC else "Outside the upper bound",
                'max_val_acc': np.around(np.amax(hist.history['val_accuracy']), decimals=3)
                if 'hist' in locals() else -3
            }
                    
        else:
            return {
                'k': 'unfeasible',
                'c': c,
                'max_val_acc' : -3
            }
    
    # the original version
    def explore_num_cells(self, k, max_atempt):
        previous_architecture = {'k': -1, 'c': -1, 'max_val_acc': -2}
        current_architecture = {'k': -1, 'c': -1, 'max_val_acc': -1}
        c = -1
        k = int(k)
        atempt = max_atempt
    
        while(current_architecture['max_val_acc'] > previous_architecture['max_val_acc']):
            previous_architecture = current_architecture
            c += 1
            self.model_counter += 1
            current_architecture = self.evaluate_model_process(k, c)
            print(f"\n\n\n{current_architecture}\n\n\n")
            
            # Give that c one more chance
            diff = previous_architecture['max_val_acc'] - current_architecture['max_val_acc']
            if atempt and diff >= 10e-4  and current_architecture['max_val_acc'] != -3:
                
                atempt -= 1
                print(f"\nRetry on (k, c) = ({k}, {c}). {atempt} chance left....\n")
                current_architecture = self.evaluate_model_process(k, c)
            
                print(f"\nAfter retrain on on (k, c) = ({k}, {c}):\n{current_architecture}\n\n")
            
        return previous_architecture

    def search(self, max_atempt=1):
        self.model_counter = 0
        epsilon = 0.005
        k0 = 4

        start = datetime.datetime.now()

        k = k0
        previous_architecture = self.explore_num_cells(k, max_atempt)
        k = 2 * k
        current_architecture = self.explore_num_cells(k, max_atempt)

        if current_architecture['max_val_acc'] > previous_architecture['max_val_acc']:
            previous_architecture = current_architecture
            k *= 2
            current_architecture = self.explore_num_cells(k, max_atempt)
            
            while(current_architecture['max_val_acc'] > previous_architecture['max_val_acc'] + epsilon):
                previous_architecture = current_architecture
                k *= 2
                current_architecture = self.explore_num_cells(k, max_atempt)
                
        else:
            k = k0 / 2
            current_architecture = self.explore_num_cells(k, max_atempt)

            while(current_architecture['max_val_acc'] >= previous_architecture['max_val_acc']):
                previous_architecture = current_architecture
                k /= 2
                current_architecture = self.explore_num_cells(k, max_atempt)

        resulting_architecture = previous_architecture

        

        if resulting_architecture['max_val_acc'] > 0:
            k = resulting_architecture['k']
            c = resulting_architecture['c']

            resulting_architecture_name = f"k_{k}_c_{c}.tflite"
            self.path_to_resulting_architecture = self.save_path / f"resulting_architecture_{resulting_architecture_name}"
            self.path_to_base_model = self.save_path / f"base_architecture_{resulting_architecture_name}"
            shutil.copy2(
                self.path_to_trained_models / resulting_architecture_name,
                self.path_to_base_model
            )
            (self.path_to_trained_models / f"{resulting_architecture_name}").rename(self.path_to_resulting_architecture)

            resulting_h5_name = f"k_{k}_c_{c}.h5"
            path_to_resulting_h5 = self.save_path / f"resulting_architecture_{resulting_h5_name}"
            (self.path_to_trained_models / resulting_h5_name).rename(path_to_resulting_h5)

            
            shutil.rmtree(self.path_to_trained_models)

            
            print(f"\nCandidate architecture: {resulting_architecture}\n")
            
            if self.two_stage:
                # ====== Now train the candidate with augmentation data =======
                print("Now train the candidate with augmentation data\n")
                self.Model(k, c)
                self.model.load_weights(path_to_resulting_h5)
                self.model.compile(
                    optimizer=tf.keras.optimizers.Adam(learning_rate=self.learning_rate),
                    loss='categorical_crossentropy',
                    metrics=['accuracy']
                )
                checkpoint = tf.keras.callbacks.ModelCheckpoint(
                    str(path_to_resulting_h5), monitor='val_accuracy',
                    verbose=0, save_best_only=True, save_weights_only=False, mode='auto'
                )
    
                hist_aug = self.model.fit(
                    self.train_ds_aug,
                    epochs=self.epochs,
                    validation_data=self.validation_ds,
                    validation_freq=1,
                    verbose=0,
                    callbacks=[checkpoint]
                )
    
                aug_val_accuracy = np.around(np.amax(hist_aug.history['val_accuracy']), decimals=3)
                print(f"\nAugmentation accuracy: {aug_val_accuracy}\n")
    
                if aug_val_accuracy > resulting_architecture['max_val_acc']:
                    print(f"Augmentation helps in case ({k}, {c})")
                    self.quantize_model_uint8(
                        src_path=path_to_resulting_h5,
                        des_path=self.path_to_resulting_architecture
                    )
                    model_type = "augmentation"
    
                else:
                    print(f"Augmentation does not help in case ({k}, {c})")
                    self.path_to_resulting_architecture = self.path_to_base_model
                    model_type = "base model"
            else:
                print("Not train the candidate with augmentation data")
                self.path_to_resulting_architecture = self.path_to_base_model                
                model_type = "base model"
            
        else:
            print(f"\nNo feasible architecture found\n")
            model_type = "No feasible architecture found"

        end = datetime.datetime.now()
        time = end - start
        print(f"Elapsed time (search): {time}\n")

        return self.path_to_resulting_architecture, self.path_to_base_model, time, model_type

Hàm `test`

In [6]:
def test_tflite_model(path_to_resulting_model, test_ds):
    interpreter = tf.lite.Interpreter(str(path_to_resulting_model))
    interpreter.allocate_tensors()

    output = interpreter.get_output_details()[0]
    input = interpreter.get_input_details()[0]

    correct = 0
    wrong = 0

    for i in test_ds:
        image, label = i[0], i[1]
        # Check if input_type is quantized, then rescale input data to uint8
        if input['dtype'] == tf.uint8:
            input_scale, input_zero_point = input['quantization']
            image = image / input_scale + input_zero_point
        input_data = tf.dtypes.cast(image, tf.uint8)
        interpreter.set_tensor(input['index'], input_data)
        interpreter.invoke()

        if label.numpy().argmax() == interpreter.get_tensor(output['index']).argmax():
            correct += 1
        else:
            wrong += 1

    print(f"{correct/(correct+wrong):.3f}")

# Thí nghiệm 1 - Chạy thí nghiệm giống bài báo gốc

## 1. Các bộ dữ liệu

### 1.1 Melanoma Skin Cancer

**Melanoma Skin Cancer** là bộ dữ liệu chứa các hình ảnh ung thư da hắc tố (melanoma) ở dạng lành tính (benign) và ác tính (malignant), được đăng tải công khai trên Kaggle tại [link](https://www.kaggle.com/datasets/hasnainjaved/melanoma-skin-cancer-dataset-of-10000-images). Mỗi ảnh được lưu ở thang màu RGB, với kích thước 300x300. Bài toán phân loại ảnh trên tập dữ liệu này yêu cầu nhận vào hình ảnh ung thư da hắc tố và đưa ra dự đoán ảnh này thuộc về lớp lành tính hay ác tính. Bộ được chia thành hai tập: tập huấn luyện với **9,605** ảnh và tập kiểm tra với **1,000 ảnh** cho cả hai lớp.

### 1.2 Tập dữ liệu Flowers-4

**Flowers-4** là một phần của bộ dữ liệu **Flowers** được đăng tải công khai
trên Kaggle tại [Drive](https://drive.google.com/file/d/18NHZUFfDyPzjsTTcTXFEtZQNC5Fesnxl/view?usp=drive_link), gồm bốn lớp: dandelion (**1,052** ảnh), iris (**1,054** ảnh), magnolia và tulip (đều **1,048** ảnh). Mỗi ảnh được lưu ở thang màu RGB, với kích thước 256x256. Bài toán phân loại ảnh trên tập dữ liệu này yêu cầu nhận vào một ảnh thuộc một trong bốn loài hoa và đưa ra dự đoán ảnh đó là loài nào. Bộ được chia thành hai tập: tập huấn luyện với **3,360** ảnh và
tập kiểm tra với **842** ảnh cho cả bốn lớp.

Note: Bộ dữ liệu Flowers gốc (https://www.kaggle.com/datasets/l3llff/flowers) hiện tại đã bị gỡ
khỏi nền tảng Kaggle. Do đó, khóa luận sử dụng bản sao của bộ dữ liệu này do tác giả A. M. Garavagno
tải về trực tiếp từ Kaggle trước đó và lưu trữ tại: [Drive](https://drive.google.com/file/d/18NHZUFfDyPzjsTTcTXFEtZQNC5Fesnxl/view?usp=drive_link)

### 1.3 Tập dữ liệu Animals-3

**Animals-3** là một phần của bộ dữ liệu **Animals-10** được đăng tải công khai trên Kaggle tại [link](https://www.kaggle.com/datasets/alessiocorrado99/animals10), gồm ba lớp: bướm (**1,689** ảnh), gà (**2,478** ảnh) và ngựa (**2,098** ảnh). Mỗi ảnh được lưu ở thang màu RGB, với kích thước đa dạng. Bài toán phân loại ảnh trên tập dữ liệu này yêu cầu nhận vào một ảnh thuộc một trong ba loài vật và đưa ra dự đoán ảnh đó là loài nào. Bộ được chia thành hai tập: tập huấn luyện với **6,265** ảnh và tập kiểm tra với **1,568** ảnh cho cả ba lớp.

### 1.4 Tập dữ liệu MNIST

Bộ dữ liệu ảnh chữ số viết tay MNIST là một bộ dữ liệu rất nổi tiếng trong lĩnh vực xây dựng mô hình phân loại ảnh. Mỗi ảnh được lưu ở thang màu xám, với kích thước 28x28. Các lớp trong bộ là các chữ số từ $0, 1, ... 9$ được phân bố đồng đều về số lượng ảnh mỗi lớp. Bộ được chia thành hai tập: tập huấn luyện với **60,000** ảnh và tập kiểm tra với **10,000** ảnh.

Reference: LeCun, Y., Cortes, C., and Burges, C. J., *MNIST handwritten digit database*, AT&T Labs [Online], Also available at https://www.kaggle.com/datasets/hojjatk/mnist-dataset, 2010. [Online].
Available: http://yann.lecun.com/exdb/mnist.

### 1.5 Tập dữ liệu Visual Wake Words

Visual Wake Words là bộ dữ liệu chứa các hình ảnh cảnh vật đời thường, được xây dựng nhằm phục vụ cho bài toán TinyML và Edge AI. Mỗi ảnh được lưu ở thang màu RGB, với kích thước đa dạng. Bài toán phân loại ảnh trên tập dữ liệu này yêu cầu nhận vào hình ảnh và đưa ra dự đoán ảnh này có ít nhất một người (nhãn “1”) hay không (nhãn “0”). Bộ được chia thành hai tập: tập huấn luyện với **115,228** ảnh và tập kiểm tra với **8,059** ảnh cho cả hai lớp.

Reference: Chowdhery, A., Warden, P., Shlens, J., Howard, A., and Rhodes, R., *Visual wake words dataset*, 2019. arXiv: 1906.05721 [cs.CV]. [Online]. Available: https://arxiv.org/abs/1906.05721.

## 2. Thí nghiệm hardware-aware trên các phần cứng hạn chế

### 2.1 Cấu hình thí nghiệm

In [8]:
hw_batch_size = 32
hw_epochs = 100
hw_input_shape = (50, 50, 3)

# Dataset directory
data_dirs = {
    'melanoma' : Path("/kaggle/input/datasets/hasnainjaved/melanoma-skin-cancer-dataset-of-10000-images/melanoma_cancer_dataset"),
    'flowers' : Path("/kaggle/input/datasets/karosvn/flowers-4/flowers"),
    'animals' : Path("/kaggle/input/datasets/karosvn/animals-3/animals"),
    'mnist' : Path("/kaggle/input/datasets/tommerfrancis/mnist-dataset/MNIST"),
    'vww' : Path("/kaggle/input/datasets/tommerfrancis/visual-wake-words/visual_wake_words")
}

# Target: STM32L010RBT6, STM32L151UCY6DTR, STM32L412KBU3
hardware_configs = {
    'L0' : {
        'RAM': 20480,
        'Flash': 131072,
        'MACC': 750000 # CoreMark * 10e4        
    },
    'L1': {
        'RAM': 32768,
        'Flash': 262144,
        'MACC': 930000 # CoreMark * 10e4
    },
    'L4': {
        'RAM': 40960,
        'Flash': 131072,
        'MACC': 2730000 # CoreMark * 10e4
    }
}

# Each dataset must comply with the following structure
# main_directory/
# ...class_a/
# ......a_image_1.jpg
# ......a_image_2.jpg
# ...class_b/
# ......b_image_1.jpg
# ......b_image_2.jpg
hw_val_split = 0.3

# whether or not to cache datasets in memory
# if the dataset cannot fit in the main memory, the application will crash
hw_cache = True

# where to save results
hw_save_path = '/kaggle/working/'

# run two-stage or not
hw_two_stage = True

### 2.2 Hàm helper cho thí nghiệm chạy `ColabNAS` trên các cấu hình phần cứng hạn chế

Hàm khởi tạo class `ColabNAS` và thực hiện thuật toán trên phần cứng `target` cụ thể

In [10]:
def searchOnHardware(target, data_dir, max_RAM, max_Flash, max_MACC, epochs,
                     val_split, cache, input_shape=(50, 50, 3), save_path='.', two_stage=False):

    print(f"\n\n\n======\t\tRun on target {target}\t\t==============\n\n\n")
    colabNAS = ColabNAS(
        max_RAM=max_RAM,
        max_Flash=max_Flash,
        max_MACC=max_MACC,
        epochs=epochs,
        path_to_training_set=data_dir / "train",
        val_split=val_split,
        cache=cache,
        input_shape=input_shape,
        save_path=save_path,
        two_stage=hw_two_stage
    )

    # Search
    path_to_resulting_model, path_to_base_model, search_time, model_type = colabNAS.search()
    
    return {
        'path_to_best_model' : path_to_resulting_model, 
        'path_to_base_model' : path_to_base_model,
        'search_time' : search_time,
        'model_type' : model_type
    }

Hàm thực hiện đánh giá mô hình trên tập `test`

In [11]:
def testModel(data_dir, path_to_resulting_model):

    test_ds = tf.keras.utils.image_dataset_from_directory(
        data_dir / "test",
        image_size=hw_input_shape[0:2],
        batch_size=hw_batch_size,
        shuffle=False
    )

    # One-hot for label
    class_names = test_ds.class_names
    num_classes = len(class_names)

    test_ds  = test_ds.map(lambda x, y: (x, tf.one_hot(y, num_classes)))
    test_ds = test_ds.unbatch().batch(1)

    for target in path_to_resulting_model.keys():
        print(f"\n===== Testing on {target} =======")
        print(f"Search time: {path_to_resulting_model[target]['search_time']}")
        print(f"Best architecture: {path_to_resulting_model[target]['path_to_best_model']}")
        print(f"Base model: {path_to_resulting_model[target]['path_to_base_model']}")
        print(f"Model type: {path_to_resulting_model[target]['model_type']}")
        base_model_path = path_to_resulting_model[target]['path_to_base_model']
        print("-- Base model accuracy:", end=" ")
        test_tflite_model(base_model_path, test_ds)
        if path_to_resulting_model[target]['model_type'] == "augmentation":
            print("-- Augmentation model accuracy:", end=" ")
            test_tflite_model(path_to_resulting_model[target]['path_to_best_model'], test_ds)


### 2.3 Chạy thí nghiệm trên từng bộ dữ liệu

Chọn tập dữ liệu từ `data_dirs`

In [12]:
# data_dir = data_dirs['melanoma'] # Tập Melanoma
# hoặc
data_dir = data_dirs['flowers'] # Tập Flowers-4
#data_dir = data_dirs['animals'] # Tập Animals-3
#data_dir = data_dirs['mnist']   # Tập MNIST
#data_dir = data_dirs['vww']     # Tập Visual Wake Words

Chạy thuật toán `ColabNAS` trên các phần cứng trong `hardware_configs`

In [13]:
path_to_resulting_model = {}

for target in hardware_configs:
    path_to_resulting_model[target] = searchOnHardware(
        target=target,
        data_dir=data_dir,
        max_RAM=hardware_configs[target]['RAM'],
        max_Flash=hardware_configs[target]['Flash'],
        max_MACC=hardware_configs[target]['MACC'],
        epochs=hw_epochs,
        val_split=hw_val_split,
        cache=hw_cache,
        save_path=hw_save_path,
        two_stage=hw_two_stage
    )




======		Run on target L0		==============





I0000 00:00:1786358319.164126      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1786358319.170180      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Found 3360 files belonging to 4 classes.
Using 2352 files for training.
Found 3360 files belonging to 4 classes.
Using 1008 files for validation.

k_4_c_0



I0000 00:00:1786358334.099572     137 service.cc:152] XLA service 0x7fbd8c026310 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1786358334.099614     137 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1786358334.099618     137 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1786358334.520857     137 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1786358336.850469     137 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Saved artifact at '/tmp/tmprk55rzqd'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 50, 50, 3), dtype=tf.float32, name='input_layer_1')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  140455499689488: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140455499688528: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140455499688720: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140455499680848: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140455499686800: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140455499688912: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140455499690448: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140455499689680: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140455499690832: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140455499690640: TensorSpec(shape=(), dtype=tf.resource, name=None)


...

W0000 00:00:1786359325.664159      57 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1786359325.664190      57 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
fully_quantize: 0, inference_type: 6, input_inference_type: UINT8, output_inference_type: UINT8



RAM: 50688,	Flash: 6184,	MACC: 1080320




{'k': 16, 'c': 0, 'RAM': 'Outside the upper bound', 'Flash': 6184, 'MACC': 1080320, 'max_val_acc': -3}




Candidate architecture: {'k': 8, 'c': 4, 'RAM': 33280, 'Flash': 33632, 'MACC': 2074856, 'max_val_acc': np.float64(0.93)}

Now train the candidate with augmentation data


Augmentation accuracy: 0.932

Augmentation helps in case (8, 4)
Saved artifact at '/tmp/tmpvqkr5d_3'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 50, 50, 3), dtype=tf.float32, name='input_layer_49')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  140449598523728: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140449598524112: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140449598527952: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140449598533712: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140449598523920: TensorSpec(shape=(), dtype=tf.r

W0000 00:00:1786359377.513475      57 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1786359377.513538      57 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.


Elapsed time (search): 0:06:01.332445



fully_quantize: 0, inference_type: 6, input_inference_type: UINT8, output_inference_type: UINT8


Test các mô hình kết quả

In [14]:
testModel(data_dir, path_to_resulting_model)

Found 842 files belonging to 4 classes.

===== Testing on L0 =======
Search time: 0:05:07.923378
Best architecture: /kaggle/working/resulting_architecture_k_2_c_4.tflite
Base model: /kaggle/working/base_architecture_k_2_c_4.tflite
Model type: augmentation
-- Base model accuracy: 

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


0.838
-- Augmentation model accuracy: 0.857

===== Testing on L1 =======
Search time: 0:06:11.930162
Best architecture: /kaggle/working/resulting_architecture_k_4_c_5.tflite
Base model: /kaggle/working/base_architecture_k_4_c_5.tflite
Model type: augmentation
-- Base model accuracy: 0.856
-- Augmentation model accuracy: 0.891

===== Testing on L4 =======
Search time: 0:06:01.332445
Best architecture: /kaggle/working/resulting_architecture_k_8_c_4.tflite
Base model: /kaggle/working/base_architecture_k_8_c_4.tflite
Model type: augmentation
-- Base model accuracy: 0.897
-- Augmentation model accuracy: 0.895


## 3. Chạy Transfer Learning

### 3.1 Cấu hình thí nghiệm

In [15]:
tl_input_shape = (224, 224, 3)

# fine-tuned MobileNetV2's params (in byte)
tl_max_RAM = 2583552
tl_max_Flash = 2713592
tl_max_MACC = 300000000 # https://arxiv.org/pdf/1801.04381.pdf

# Each dataset must comply with the following structure
# main_directory/
# ...class_a/
# ......a_image_1.jpg
# ......a_image_2.jpg
# ...class_b/
# ......b_image_1.jpg
# ......b_image_2.jpg
tl_val_split = 0.3

# whether or not to cache datasets in memory
# if the dataset cannot fit in the main memory, the application will crash
tl_cache = True

# where to save results
tl_save_path = '/kaggle/working/'

tl_batch_size = 128
epochs_transfer_learning = 20
epochs_fine_tuning = 10
epochs_colabnas = 100

### 3.2 Hàm helper chạy thí nghiệm so sánh với Transfer Learning

In [16]:
# load and preprocess image dataset for transfer learning model training and evaluation
def load_dataset(path_to_training_set, path_to_test_set, batch_size, validation_split, input_shape):
    num_classes = len(next(os.walk(path_to_training_set))[1])
    
    train_ds = tf.keras.utils.image_dataset_from_directory(
        directory = path_to_training_set,
        labels = "inferred",
        label_mode = "categorical",
        color_mode = "rgb",
        batch_size = batch_size,
        image_size = (input_shape[0], input_shape[1]),
        shuffle = True,
        seed = FIXED_SEED,
        validation_split = validation_split,
        subset = "training"
    )

    validation_ds = tf.keras.utils.image_dataset_from_directory(
        directory = path_to_training_set,
        labels = "inferred",
        label_mode = "categorical",
        color_mode = "rgb",
        batch_size = batch_size,
        image_size = (input_shape[0], input_shape[1]),
        shuffle = True,
        seed = FIXED_SEED,
        validation_split = validation_split,
        subset = "validation"
    )

    test_ds = tf.keras.utils.image_dataset_from_directory(
        directory = path_to_test_set,
        labels = "inferred",
        label_mode = "categorical",
        color_mode = "rgb",
        batch_size = batch_size,
        image_size = (input_shape[0], input_shape[1]),
        shuffle = False,
        #seed = 11
    )

    # cache train and validation sets in memory (RAM)
    AUTOTUNE = tf.data.AUTOTUNE
    train_ds = train_ds.cache().prefetch(buffer_size = AUTOTUNE)
    validation_ds = validation_ds.cache().prefetch(buffer_size = AUTOTUNE)
    test_ds = test_ds.prefetch(buffer_size = AUTOTUNE)

    return train_ds, validation_ds, test_ds, num_classes

def quantize_model_uint8(train_ds, model_name): # apply post training quantization to transfer learning model
    def representative_dataset(): # provide small size of samples for calibration to determine activation ranges for accurate int8 quantization
        for data in train_ds.rebatch(1).take(150):
            yield [tf.dtypes.cast(data[0], tf.float32)]

    model = tf.keras.models.load_model(f"{model_name}.h5")
    converter = tf.lite.TFLiteConverter.from_keras_model(model) # convert model to TensorFlow Lite
    converter.optimizations = [tf.lite.Optimize.DEFAULT] # enable int8 quantization
    converter.representative_dataset = representative_dataset
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8] # force strictly int8 operations
    converter.inference_input_type = tf.uint8
    converter.inference_output_type = tf.uint8
    tflite_quant_model = converter.convert()

    with open(f"{model_name}.tflite", "wb") as f:
        f.write(tflite_quant_model)
    os.remove(f"{model_name}.h5")


Class `TLModel`

In [17]:
class TLModel:
    def __init__(self, input_shape, num_classes, 
                 epochs_train, epochs_finetune, save_path='.'):
        
        self.base_model = tf.keras.applications.MobileNetV2( # MobileNetV2 is used as backbone
            weights = "imagenet", # load weights pre-trained on ImageNet
            input_shape = tl_input_shape, 
            include_top = False) # drop original 1000-class classification layer of MobileNetV2
        
        self.base_model.trainable = False # freeze CNN layers

        #self.initializer = initializer
        self.input_shape = input_shape
        self.num_classes = num_classes
        self.epochs_train = epochs_train
        self.epochs_finetune = epochs_finetune
        self.save_path = save_path

        # auto-save model with highest validation accuracy during training
        self.checkpoint = tf.keras.callbacks.ModelCheckpoint(
            save_path + ".h5", 
            monitor="val_accuracy", 
            verbose=0,
            save_best_only=True, 
            save_weights_only=False, 
            mode="auto"
        )
        

    def pipeline(self):

        inputs = tf.keras.Input(shape=self.input_shape)

        # pre-processing pipeline
        x = tf.keras.layers.RandomFlip("horizontal")(inputs) # horizontal flips
        x = tf.keras.layers.RandomRotation(factor = 0.2, fill_mode = "constant", interpolation = "bilinear")(x)
        x = tf.keras.layers.Rescaling(1/255)(x) # min-max standardization
        x = tf.keras.layers.BatchNormalization()(x)

        # The base model contains batchnorm layers. 
        # We want to keep them in inference mode when we unfreeze the base model for fine-tuning,
        # so we make sure that the base model is running in inference mode here.
        x = self.base_model(x, training=False)
        
        # custom classifier
        x = tf.keras.layers.GlobalAveragePooling2D()(x)

        # single fully connected layer
        outputs = tf.keras.layers.Dense(self.num_classes, activation = "softmax")(x) 
        # outputs = tf.keras.layers.Dense(self.num_classes, activation = "softmax", kernel_initializer=self.initializer)(x) 
        model = tf.keras.Model(inputs=inputs, outputs=outputs)

        # optimizer and compile
        optimizer = tf.keras.optimizers.Adam(learning_rate = 1e-3)
        model.compile(
            optimizer=optimizer, 
            loss="categorical_crossentropy",
            metrics=["accuracy"])
        
        self.model = model
        model.summary()

    def train_classifier(self, train_ds, val_ds):
        # train custom classifier
        self.model.fit(
            x=train_ds, 
            epochs=self.epochs_train, 
            validation_data=val_ds, 
            verbose=0,
            validation_freq=1)
        
        self.base_model.trainable = True # unfreeze CNN layers to fine-tune model

    def finetune(self, train_ds, val_ds):
        # optimizer and compile
        optimizer = tf.keras.optimizers.Adam(learning_rate=1e-5) 
        self.model.compile(
            optimizer=optimizer, 
            loss="categorical_crossentropy", 
            metrics=["accuracy"])

        # fine-tune model
        self.model.fit(
            x=train_ds, 
            epochs=self.epochs_finetune,
            validation_data=val_ds, 
            callbacks = [self.checkpoint], 
            verbose=0, 
            validation_freq = 1)
        
    def evaluate(self, test_ds):
        self.model.load_weights(self.save_path + ".h5") # load best weights
        test_acc = self.model.evaluate(test_ds, return_dict=True) # evaluate model on test dataset
        
        return test_acc


### 3.2 Chạy thí nghiệm

Chọn tập dữ liệu từ `data_dirs`

In [18]:
#data_dir = data_dirs['melanoma'] # Tập Melanoma
# hoặc
data_dir = data_dirs['flowers'] # Tập Flowers-4
# data_dir = data_dirs['animals'] # Tập Animals-3

#### Transfer Learning

In [19]:
# # [Fix Randomness] Clear the previous Keras session to provide a clean state before initializing the random seed
# tf.keras.backend.clear_session()
# # [Fix Randomness] Set global seed for reproducible results across Python, NumPy, and Keras
# tf.keras.utils.set_random_seed(FIXED_SEED)
# # [Fix Randomness] Enable deterministic operations to prevent GPU floating-point variations
# tf.config.experimental.enable_op_determinism()
# # [Fix Randomness] Fix random seed for weight initialization to guarantee consistent model convergence
# initializer = tf.keras.initializers.GlorotUniform(seed=FIXED_SEED)

# Load dataset
train_ds, val_ds, test_ds, num_classes = load_dataset(
    path_to_training_set=data_dir / "train", 
    path_to_test_set=data_dir / "test", 
    batch_size=tl_batch_size, 
    validation_split=tl_val_split, 
    input_shape=tl_input_shape)

# START TIMER
start = datetime.datetime.now() 

# init Transfer Learning model
tl_model = TLModel(
    #initializer=initializer,
    input_shape=tl_input_shape,
    num_classes=num_classes,
    epochs_train=epochs_transfer_learning,
    epochs_finetune=epochs_fine_tuning,
    save_path=tl_save_path
)

# create model pipeline
tl_model.pipeline()

# train classifier
tl_model.train_classifier(train_ds, val_ds)

# fine-tune model
tl_model.finetune(train_ds, val_ds)

# STOP TIMER
end = datetime.datetime.now() 
print(f"\nTraining time: {end - start}\n") 

Found 3360 files belonging to 4 classes.
Using 2352 files for training.
Found 3360 files belonging to 4 classes.
Using 1008 files for validation.
Found 842 files belonging to 4 classes.
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional_50"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_51 (InputLayer)     │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_flip_3 (RandomFlip)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_rotation_3               │ (None, 224, 224, 3)    │             0 │
│ (RandomRotation)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling_47 (Rescaling)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_47          │ (None, 224, 224, 3)    │            12 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_47     │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_94 (Dense)                │ (None, 4)              │         5,124 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,263,120 (8.63 MB)

 Trainable params: 5,130 (20.04 KB)

 Non-trainable params: 2,257,990 (8.61 MB)


Training time: 0:06:52.400298



#### ColabNAS

In [20]:
# initialize ColabNAS with MobileNetV2 constraints
colabNAS = ColabNAS(
    max_RAM=tl_max_RAM, 
    max_Flash=tl_max_Flash, 
    max_MACC=tl_max_MACC,
    path_to_training_set=data_dir / "train", 
    val_split=tl_val_split, 
    epochs=epochs_colabnas,
    cache=tl_cache, 
    input_shape=tl_input_shape, 
    save_path=tl_save_path)

# search
path_to_resulting_architecture, path_to_base_model, time, model_type = colabNAS.search(max_atempt=0)

Found 3360 files belonging to 4 classes.
Using 2352 files for training.
Found 3360 files belonging to 4 classes.
Using 1008 files for validation.

k_4_c_0



2026-08-10 11:03:31.571507: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-08-10 11:03:31.850401: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


Saved artifact at '/tmp/tmpi3pqqty9'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_layer_53')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  140449812297104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140449812296336: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140449812296528: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140449812297488: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140449812298064: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140449812298448: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140449812296720: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140449812290192: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140449812298256: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140449812289040: TensorSpec(shape=(), dtype=tf.resource, name=None)


W0000 00:00:1786359821.635628      57 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1786359821.635681      57 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
fully_quantize: 0, inference_type: 6, input_inference_type: UINT8, output_inference_type: UINT8



RAM: 354304,	Flash: 4896,	MACC: 5419040

Saved artifact at '/tmp/tmpcam23oom'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_layer_53')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  140449592536976: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140449592536592: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140449809976016: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140449809975248: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140449592538704: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140449592528528: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140449592539088: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140449592536784: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140449592529104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140449592540432: TensorSpec(shape=(), dt

...

W0000 00:00:1786363294.353887      57 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1786363294.353946      57 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
fully_quantize: 0, inference_type: 6, input_inference_type: UINT8, output_inference_type: UINT8


Best val_accuracy: 0.911 at epoch 97/99



{'k': 32, 'c': 1, 'RAM': 2011648, 'Flash': 33496, 'MACC': 274567424, 'max_val_acc': np.float64(0.911)}




k_32_c_2

Saved artifact at '/tmp/tmpzv_tu2g4'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_layer_75')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  140449502598864: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140449502600976: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140449502593296: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140449502600016: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140449502598288: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140449502596752: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140449502600208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140449502599440: TensorSpec(shape=(), dtype=tf.resource, name=

W0000 00:00:1786363317.554078      57 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1786363317.554101      57 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.



RAM: 2013184,	Flash: 98424,	MACC: 447980928




{'k': 32, 'c': 2, 'RAM': 2013184, 'Flash': 98424, 'MACC': 'Outside the upper bound', 'max_val_acc': -3}




Candidate architecture: {'k': 16, 'c': 4, 'RAM': 1010688, 'Flash': 102344, 'MACC': 150354144, 'max_val_acc': np.float64(0.962)}

Not train the candidate with augmentation data
Elapsed time (search): 0:58:38.559182



fully_quantize: 0, inference_type: 6, input_inference_type: UINT8, output_inference_type: UINT8


Test ColabNAS and Transfer Learning

In [21]:
# Test TL model
print("\n========Transfer Learning========\n")
test_acc = tl_model.evaluate(test_ds)
print(f"Test Accuracy: \n{test_acc}")

# Test ColabNAS
print("\n========ColabNAS========\n")
test_ds = tf.keras.utils.image_dataset_from_directory(
    directory = data_dir / "test",
    labels = "inferred",
    label_mode = "categorical",
    color_mode = "rgb",
    batch_size = tl_batch_size,
    image_size = tl_input_shape[0:2],
    shuffle = False,
    #seed = 
)

test_ds = test_ds.unbatch().batch(1)

print(f"\n===== Testing on ColabNAS =======")
print(f"Search time: {time}")
print(f"Best architecture: {path_to_resulting_architecture}")
print(f"Base model: {path_to_base_model}")
print(f"Model type: {model_type}")
print("-- Base model accuracy:", end=" ")
test_tflite_model(path_to_base_model, test_ds)
if path_to_resulting_model[target]['model_type'] == "augmentation":
    print("-- Augmentation model accuracy:", end=" ")
    test_tflite_model(path_to_resulting_architecture, test_ds)


========Transfer Learning========

7/7 ━━━━━━━━━━━━━━━━━━━━ 3s 337ms/step - accuracy: 0.9808 - loss: 0.0592
Test Accuracy: 
{'accuracy': 0.9774346947669983, 'loss': 0.07203321158885956}

========ColabNAS========

Found 842 files belonging to 4 classes.

===== Testing on ColabNAS =======
Search time: 0:58:38.559182
Best architecture: /kaggle/working/base_architecture_k_16_c_4.tflite
Base model: /kaggle/working/base_architecture_k_16_c_4.tflite
Model type: base model
-- Base model accuracy: 0.936
-- Augmentation model accuracy: 0.936


## 4. Chạy thí nghiệm so sánh ColabNAS với SOTA trên tập VWW

Chọn tập dữ liệu từ `data_dirs`

In [9]:
data_dir = data_dirs['vww'] 

### 4.1 Cấu hình thí nghiệm

In [10]:
sota_input_shape = (50, 50, 3)

# target: STMF446RE
sota_max_RAM = 131072
sota_max_Flash = 524288
sota_max_MACC = 6080000 # CoreMark * 10^4

# Each dataset must comply with the following structure
# main_directory/
# ...class_a/
# ......a_image_1.jpg
# ......a_image_2.jpg
# ...class_b/
# ......b_image_1.jpg
# ......b_image_2.jpg
sota_val_split = 0.3
sota_batch_size = 32
sota_epochs = 100

# whether or not to cache datasets in memory
# if the dataset cannot fit in the main memory, the application will crash
sota_cache = True

# where to save results
sota_save_path = '/kaggle/working/'

sota_two_stage = False
sota_max_atempt = 0

### 4.2 ColabNAS

In [11]:
# initialize ColabNAS with STMF446RE constraints
colabNAS = ColabNAS(
    max_RAM=sota_max_RAM, 
    max_Flash=sota_max_Flash, 
    max_MACC=sota_max_MACC,
    epochs=sota_epochs,
    path_to_training_set=data_dir / "train", 
    val_split=sota_val_split, 
    cache=sota_cache, 
    input_shape=sota_input_shape, 
    save_path=sota_save_path,
    two_stage=sota_two_stage
)

# search
path_to_resulting_architecture, path_to_base_model, time, model_type = colabNAS.search(max_atempt=sota_max_atempt)

I0000 00:00:1786373068.692024      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1786373068.698080      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Found 115228 files belonging to 2 classes.
Using 80660 files for training.
Found 115228 files belonging to 2 classes.
Using 34568 files for validation.

k_4_c_0



I0000 00:00:1786373364.839056     146 service.cc:152] XLA service 0x7ce19c0057d0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1786373364.839100     146 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1786373364.839105     146 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1786373365.252663     146 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1786373367.621746     146 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Saved artifact at '/tmp/tmpno5cf0w6'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 50, 50, 3), dtype=tf.float32, name='input_layer_1')
Output Type:
  TensorSpec(shape=(None, 2), dtype=tf.float32, name=None)
Captures:
  137311721567248: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137311721566288: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137311721565712: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137311721558416: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137311721564368: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137311721566096: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137311721567824: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137311721567440: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137311721568208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137311721568016: TensorSpec(shape=(), dtype=tf.resource, name=None)


W0000 00:00:1786373659.844329      57 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1786373659.844351      57 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
I0000 00:00:1786373659.850573      57 mlir_graph_optimization_pass.cc:425] MLIR V1 optimization pass is not enabled
fully_quantize: 0, inference_type: 6, input_inference_type: UINT8, output_inference_type: UINT8


...

W0000 00:00:1786377875.676080      57 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1786377875.676106      57 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
fully_quantize: 0, inference_type: 6, input_inference_type: UINT8, output_inference_type: UINT8


Best val_accuracy: 0.722 at epoch 99/99



{'k': 32, 'c': 0, 'RAM': 91136, 'Flash': 8216, 'MACC': 2161088, 'max_val_acc': np.float64(0.722)}




k_32_c_1

Saved artifact at '/tmp/tmpyxq29fai'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 50, 50, 3), dtype=tf.float32, name='input_layer_14')
Output Type:
  TensorSpec(shape=(None, 2), dtype=tf.float32, name=None)
Captures:
  137307760400080: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137307760399888: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137307886680016: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137307760402000: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137307760394320: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137307760397968: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137307760399312: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137307760399504: TensorSpec(shape=(), dtype=tf.resource, name=None)
 

W0000 00:00:1786377888.836043      57 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1786377888.836070      57 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.



RAM: 104448,	Flash: 33296,	MACC: 13684224




{'k': 32, 'c': 1, 'RAM': 104448, 'Flash': 33296, 'MACC': 'Outside the upper bound', 'max_val_acc': -3}




Candidate architecture: {'k': 16, 'c': 2, 'RAM': 54784, 'Flash': 31384, 'MACC': 5953056, 'max_val_acc': np.float64(0.787)}

Not train the candidate with augmentation data
Elapsed time (search): 1:15:27.087509



fully_quantize: 0, inference_type: 6, input_inference_type: UINT8, output_inference_type: UINT8


Test ColabNAS

In [13]:
# Test ColabNAS
print("\n========ColabNAS========\n")
test_ds = tf.keras.utils.image_dataset_from_directory(
    directory = data_dir / "test",
    labels = "inferred",
    label_mode = "categorical",
    color_mode = "rgb",
    batch_size = sota_batch_size,
    image_size = sota_input_shape[0:2],
    shuffle = False,
    #seed = 
)

test_ds = test_ds.unbatch().batch(1)
print(f"\n===== Testing on ColabNAS =======")
print(f"Search time: {time}")
print(f"Best architecture: {path_to_resulting_architecture}")
print(f"Base model: {path_to_base_model}")
print(f"Model type: {model_type}")
print("-- Base model accuracy:", end=" ")
test_tflite_model(path_to_base_model, test_ds)
if model_type == "augmentation":
    print("-- Augmentation model accuracy:", end=" ")
    test_tflite_model(path_to_resulting_architecture, test_ds)


========ColabNAS========

Found 8059 files belonging to 2 classes.

===== Testing on ColabNAS =======
Search time: 1:15:27.087509
Best architecture: /kaggle/working/base_architecture_k_16_c_2.tflite
Base model: /kaggle/working/base_architecture_k_16_c_2.tflite
Model type: base model
-- Base model accuracy: 0.771


In [14]:
!/kaggle/working/stm32tflm $path_to_tflite_model # run TFLite model on STM32TFLM simulator

/kaggle/working/stm32tflm <tflite model>


In [15]:
interpreter = tf.lite.Interpreter(str(path_to_resulting_architecture)) # load optimal ColabNAS model into TFLite interpreter for evaluation
interpreter.allocate_tensors() # allocate memory for model's tensors before evaluation
%timeit interpreter.invoke() # measure average execution time of optimal ColabNAS model

387 µs ± 11.9 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)
